# Quantitative Finance Formula Cheat Sheet

A comprehensive, beginner-friendly notebook covering 8 core areas of quantitative finance.
Each formula is explained in plain English, implemented in Python, and demonstrated with a concrete numeric example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq

np.random.seed(42)

---
## 1. VOLATILITY
---

### Realized Volatility

Realized volatility measures how much an asset's returns have actually varied over a past period.
It is the standard deviation of historical returns. Higher realized volatility means the asset's price
has been swinging more wildly, which generally implies higher risk.

In [ ]:
def realized_volatility(returns: np.ndarray) -> float:
    """Compute the realized (historical) volatility as the sample standard deviation of returns."""
    n = len(returns)
    mean_return = np.mean(returns)
    variance = np.sum((returns - mean_return) ** 2) / (n - 1)
    return np.sqrt(variance)

In [ ]:
daily_returns = np.array([0.01, -0.005, 0.008, -0.003, 0.012, -0.007, 0.004, 0.002, -0.001, 0.006])
vol = realized_volatility(daily_returns)
print(f"The realized daily volatility is {vol:.6f} ({vol*100:.4f}%), meaning the asset's daily returns "
      f"typically deviate by about {vol*100:.4f} percentage points from the average.")
annualized = vol * np.sqrt(252)
print(f"Annualized (assuming 252 trading days), that corresponds to roughly {annualized*100:.2f}% volatility.")

### Implied Volatility

Implied volatility is the market's forecast of future volatility, extracted from the price of an option.
We observe the market price of a call option and then work backwards through the Black-Scholes formula
to find what volatility would produce that price. It uses a numerical root-finding method (Brent's method)
because there is no closed-form inverse.

In [ ]:
def black_scholes_call(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Compute Black-Scholes call option price."""
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def implied_volatility(market_price: float, S: float, K: float, T: float, r: float) -> float:
    """Solve for implied volatility using Brent's method."""
    objective = lambda sigma: black_scholes_call(S, K, T, r, sigma) - market_price
    return brentq(objective, 1e-6, 5.0)

In [ ]:
S, K, T, r = 100.0, 105.0, 0.5, 0.05
market_price = 5.50
iv = implied_volatility(market_price, S, K, T, r)
print(f"Given a market call price of ${market_price:.2f} (S={S}, K={K}, T={T}y, r={r}), "
      f"the implied volatility is {iv*100:.2f}%.")
print(f"This means the market expects the stock to fluctuate by roughly {iv*100:.2f}% per year.")

### Forward Volatility

Forward volatility is the expected volatility over a future time window (from T1 to T2),
implied by the term structure of volatility. If you know the total variance up to time T1 and T2,
you can extract the variance that belongs to the interval in between. It is useful for pricing
options that start in the future.

In [ ]:
def forward_volatility(sigma_T1: float, T1: float, sigma_T2: float, T2: float) -> float:
    """Compute forward volatility between T1 and T2 from the volatility term structure."""
    forward_variance = (sigma_T2**2 * T2 - sigma_T1**2 * T1) / (T2 - T1)
    return np.sqrt(forward_variance)

In [ ]:
sigma_1y, sigma_2y = 0.20, 0.25
fwd_vol = forward_volatility(sigma_1y, 1.0, sigma_2y, 2.0)
print(f"If the 1-year implied vol is {sigma_1y*100:.0f}% and the 2-year implied vol is {sigma_2y*100:.0f}%, "
      f"the forward volatility for year 1-to-2 is {fwd_vol*100:.2f}%.")
print(f"This tells us the market expects higher uncertainty in the second year than the first.")

### Cumulative Return

Cumulative return tells you the total growth (or loss) of an investment over multiple periods.
Instead of adding returns, you multiply the growth factors (1 + r) together, because each period's
return compounds on the previous balance. A cumulative return of 0.05 means you gained 5% overall.

In [ ]:
def cumulative_return(returns: np.ndarray) -> float:
    """Compute the cumulative (compounded) return from a series of periodic returns."""
    return np.prod(1 + returns) - 1

In [ ]:
monthly_returns = np.array([0.02, -0.01, 0.03, 0.005, -0.02, 0.015])
cum_ret = cumulative_return(monthly_returns)
print(f"Over 6 months with returns {list(monthly_returns)}, the cumulative return is {cum_ret*100:.2f}%.")
print(f"An initial $10,000 investment would now be worth ${10000*(1+cum_ret):,.2f}.")

---
## 2. RISK METRICS
---

### Value at Risk (VaR) — Parametric

Value at Risk answers: "What is the worst loss I can expect over a given period at a given
confidence level?" For example, a 1-day 95% VaR of $10,000 means there is only a 5% chance
of losing more than $10,000 in a single day. The parametric method assumes returns are normally distributed.

In [ ]:
def var_parametric(mu: float, sigma: float, confidence: float = 0.95) -> float:
    """Compute parametric Value at Risk at the given confidence level."""
    z_alpha = norm.ppf(confidence)
    return mu - z_alpha * sigma

In [ ]:
mu_daily, sigma_daily = 0.0005, 0.02
var_95 = var_parametric(mu_daily, sigma_daily, 0.95)
portfolio_value = 1_000_000
print(f"The 95% daily VaR is {var_95*100:.4f}% of portfolio value.")
print(f"For a ${portfolio_value:,} portfolio, this means a maximum expected daily loss of "
      f"${abs(var_95)*portfolio_value:,.2f} at the 95% confidence level.")

### Sharpe Ratio

The Sharpe ratio measures risk-adjusted return: how much excess return you earn per unit of risk.
A Sharpe of 1.0 means you earn 1 unit of return for every 1 unit of volatility you endure.
Higher is better. A Sharpe below 0 means you'd have been better off in a risk-free asset.

In [ ]:
def sharpe_ratio(expected_return: float, risk_free_rate: float, volatility: float) -> float:
    """Compute the Sharpe ratio: excess return per unit of total risk."""
    return (expected_return - risk_free_rate) / volatility

In [ ]:
E_R, Rf, sigma = 0.12, 0.04, 0.15
sr = sharpe_ratio(E_R, Rf, sigma)
print(f"With an expected return of {E_R*100:.0f}%, risk-free rate of {Rf*100:.0f}%, "
      f"and volatility of {sigma*100:.0f}%, the Sharpe ratio is {sr:.2f}.")
print(f"This means the portfolio earns {sr:.2f} units of excess return per unit of risk taken.")

### Sortino Ratio

The Sortino ratio is like the Sharpe ratio, but it only penalizes downside risk (losses).
Investors generally don't mind upside volatility — it's the drops that hurt. The Sortino ratio
uses the standard deviation of only negative returns, giving a more realistic picture of
risk-adjusted performance for portfolios with asymmetric return distributions.

In [ ]:
def sortino_ratio(returns: np.ndarray, risk_free_rate: float) -> float:
    """Compute the Sortino ratio using downside deviation."""
    excess_return = np.mean(returns) - risk_free_rate
    downside_returns = returns[returns < 0]
    downside_std = np.std(downside_returns, ddof=1)
    return excess_return / downside_std

In [ ]:
returns_sample = np.array([0.03, -0.01, 0.05, -0.02, 0.04, -0.005, 0.02, -0.015, 0.01, 0.035])
rf_monthly = 0.003
sort_r = sortino_ratio(returns_sample, rf_monthly)
print(f"The Sortino ratio is {sort_r:.2f}, meaning the portfolio earns {sort_r:.2f} units of "
      f"excess return per unit of downside risk.")
print(f"Compared to Sharpe, Sortino is more forgiving of upside volatility.")

### RAROC (Risk-Adjusted Return on Capital)

RAROC measures how much return a business unit or trade generates relative to the amount of
capital set aside to cover potential losses (economic capital). Banks use it to compare the
profitability of different activities on a level playing field, since riskier activities require
more capital reserves.

In [ ]:
def raroc(expected_return: float, economic_capital: float) -> float:
    """Compute Risk-Adjusted Return on Capital."""
    return expected_return / economic_capital

In [ ]:
exp_ret, eco_cap = 500_000, 2_000_000
r = raroc(exp_ret, eco_cap)
print(f"With an expected return of ${exp_ret:,} and economic capital of ${eco_cap:,}, "
      f"the RAROC is {r*100:.1f}%.")
print(f"This means the business unit generates {r*100:.1f} cents of return for every dollar of risk capital.")

---
## 3. OPTIONS & GREEKS
---

### Black-Scholes Call Price

The Black-Scholes formula is the cornerstone of options pricing. It gives the theoretical fair
price of a European call option — the right to buy a stock at a fixed price (strike) at a future
date. The formula accounts for the current stock price, the strike, time to expiry, the risk-free
interest rate, and the stock's volatility.

In [ ]:
# black_scholes_call was already defined above in the Implied Volatility section.
# Repeating here for clarity and self-containedness.

def bs_call(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Compute the Black-Scholes European call option price."""
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

In [ ]:
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
price = bs_call(S, K, T, r, sigma)
print(f"A European call option with S=${S}, K=${K}, T={T}y, r={r*100}%, sigma={sigma*100}% "
      f"is worth ${price:.2f}.")
print(f"This is the fair premium you'd pay today for the right to buy the stock at ${K} in {T} year.")

### Delta

Delta measures how much the option price changes when the underlying stock moves by $1.
A delta of 0.6 means the option gains about $0.60 for every $1 the stock goes up.
Delta also roughly represents the probability that the option will expire in-the-money.

In [ ]:
def bs_delta(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Compute the Black-Scholes delta of a European call option."""
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1)

In [ ]:
delta = bs_delta(S, K, T, r, sigma)
print(f"The delta of this at-the-money call is {delta:.4f}.")
print(f"If the stock rises by $1, the option price increases by approximately ${delta:.2f}.")

### Gamma

Gamma measures how fast delta itself changes as the stock price moves. It is the second
derivative of the option price with respect to the stock price. High gamma means delta is
very sensitive to stock moves — this happens when the option is near the money and close to expiry.

In [ ]:
def bs_gamma(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Compute the Black-Scholes gamma of a European call option."""
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

In [ ]:
gamma = bs_gamma(S, K, T, r, sigma)
print(f"The gamma is {gamma:.6f}.")
print(f"This means that if the stock moves $1, delta changes by approximately {gamma:.4f}.")

### Vega

Vega measures how sensitive the option price is to changes in implied volatility.
If vega is 15, the option price rises by about $0.15 for every 1 percentage-point increase
in implied volatility. Vega is highest for at-the-money options with lots of time remaining.

In [ ]:
def bs_vega(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Compute the Black-Scholes vega of a European call option."""
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)

In [ ]:
vega = bs_vega(S, K, T, r, sigma)
print(f"The vega is {vega:.2f}.")
print(f"If implied volatility rises by 1 percentage point (e.g., 20% to 21%), "
      f"the option price increases by approximately ${vega * 0.01:.2f}.")

---
## 4. CREDIT RISK & EXECUTION
---

### Loan-to-Value (LTV)

LTV is the ratio of a loan amount to the value of the collateral backing it.
An LTV of 0.80 means the borrower owes 80% of the asset's value. Higher LTV means more
risk for the lender — if the collateral drops in value, the loan may become under-collateralized.

In [ ]:
def loan_to_value(loan: float, collateral_value: float) -> float:
    """Compute the loan-to-value ratio."""
    return loan / collateral_value

In [ ]:
loan, collateral = 320_000, 400_000
ltv = loan_to_value(loan, collateral)
print(f"A loan of ${loan:,} against collateral worth ${collateral:,} gives an LTV of {ltv*100:.0f}%.")
print(f"The borrower has {(1-ltv)*100:.0f}% equity cushion before the loan is underwater.")

### Expected Loss

Expected loss is the average amount a lender expects to lose from a loan. It combines three
factors: the probability the borrower defaults (PD), the fraction of the loan lost if default
happens (LGD), and the total amount at risk (EAD). Banks use EL to set aside loan loss reserves.

In [ ]:
def expected_loss(pd: float, lgd: float, ead: float) -> float:
    """Compute expected loss: PD x LGD x EAD."""
    return pd * lgd * ead

In [ ]:
pd, lgd, ead = 0.02, 0.45, 1_000_000
el = expected_loss(pd, lgd, ead)
print(f"With PD={pd*100}%, LGD={lgd*100}%, and EAD=${ead:,}, the expected loss is ${el:,.0f}.")
print(f"The bank should reserve at least ${el:,.0f} against this loan on average.")

### TWAP (Time-Weighted Average Price)

TWAP is the simple average of prices sampled at equal time intervals. Traders use TWAP algorithms
to spread a large order evenly across a time period, reducing the chance that a single large
trade moves the market price against them.

In [ ]:
def twap(prices: np.ndarray) -> float:
    """Compute the time-weighted average price."""
    return np.mean(prices)

In [ ]:
prices = np.array([100.5, 101.0, 100.8, 101.5, 102.0, 101.2])
t = twap(prices)
print(f"The prices over 6 intervals were {list(prices)}.")
print(f"The TWAP is ${t:.2f}, representing the fair average price across the time window.")

### VWAP (Volume-Weighted Average Price)

VWAP weights each price by the volume traded at that price, so periods with heavy trading
count more. It is the benchmark most institutional traders use — if you buy below VWAP,
you did better than the average market participant that day.

In [ ]:
def vwap(prices: np.ndarray, volumes: np.ndarray) -> float:
    """Compute the volume-weighted average price."""
    return np.sum(prices * volumes) / np.sum(volumes)

In [ ]:
prices = np.array([100.5, 101.0, 100.8, 101.5, 102.0, 101.2])
volumes = np.array([1000, 1500, 800, 2000, 500, 1200])
v = vwap(prices, volumes)
print(f"The VWAP is ${v:.2f}, compared to the simple average (TWAP) of ${np.mean(prices):.2f}.")
print(f"The VWAP is slightly different because more volume traded at the higher price levels.")

---
## 5. STOCHASTIC PROCESSES
---

### Brownian Motion

Brownian motion (or Wiener process) is the mathematical foundation of random walks in finance.
Each step is an independent, normally distributed random increment. Stock prices are often
modeled as driven by Brownian motion — it captures the unpredictable, continuous nature of
market movements.

In [ ]:
def simulate_brownian_motion(n_steps: int, dt: float) -> np.ndarray:
    """Simulate a standard Brownian motion path with n_steps increments of size dt."""
    dW = np.random.normal(0, np.sqrt(dt), size=n_steps)
    W = np.concatenate([[0], np.cumsum(dW)])
    return W

In [ ]:
n_steps, dt = 1000, 0.001
W = simulate_brownian_motion(n_steps, dt)
t = np.linspace(0, n_steps * dt, n_steps + 1)

plt.figure(figsize=(10, 4))
plt.plot(t, W)
plt.title("Standard Brownian Motion")
plt.xlabel("Time")
plt.ylabel("W(t)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Simulated {n_steps} steps of Brownian motion over T={n_steps*dt:.1f}.")
print(f"The final value W(T) = {W[-1]:.4f}, which is random and centered around 0.")

### Ito Process

An Ito process adds drift (a trend) and diffusion (randomness) to Brownian motion.
The equation dX = mu*dt + sigma*dW means the process has a deterministic drift of mu per
unit time, plus random fluctuations scaled by sigma. Most asset price models in finance
are Ito processes. We simulate it using the Euler-Maruyama method.

In [ ]:
def simulate_ito_process(x0: float, mu: float, sigma: float, dt: float, n_steps: int) -> np.ndarray:
    """Simulate an Ito process using the Euler-Maruyama method."""
    X = np.zeros(n_steps + 1)
    X[0] = x0
    for i in range(n_steps):
        dW = np.random.normal(0, np.sqrt(dt))
        X[i + 1] = X[i] + mu * dt + sigma * dW
    return X

In [ ]:
x0, mu, sigma, dt, n_steps = 10.0, 0.5, 1.0, 0.01, 1000
X = simulate_ito_process(x0, mu, sigma, dt, n_steps)
t = np.linspace(0, n_steps * dt, n_steps + 1)

plt.figure(figsize=(10, 4))
plt.plot(t, X, label="Ito Process")
plt.plot(t, x0 + mu * t, '--', color='red', label=f"Drift (mu={mu})")
plt.title("Ito Process: dX = μdt + σdW")
plt.xlabel("Time")
plt.ylabel("X(t)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Starting at X0={x0}, with drift mu={mu} and volatility sigma={sigma}, "
      f"the process ended at X(T)={X[-1]:.2f} after T={n_steps*dt:.0f} time units.")

### Ito's Lemma

Ito's Lemma is the chain rule of stochastic calculus. If you have a function f(X) of a
stochastic process X, Ito's Lemma tells you how f(X) evolves. Unlike the regular chain rule,
it has an extra term (the σ²/2 correction) because random processes have non-zero quadratic
variation. We demonstrate with f(x) = x².

In [ ]:
def demonstrate_itos_lemma(x0: float, mu: float, sigma: float, dt: float, n_steps: int) -> tuple:
    """Demonstrate Ito's Lemma for f(x) = x², comparing direct vs Ito computation."""
    # Simulate X_t
    X = np.zeros(n_steps + 1)
    X[0] = x0
    dW_all = np.random.normal(0, np.sqrt(dt), size=n_steps)
    for i in range(n_steps):
        X[i + 1] = X[i] + mu * dt + sigma * dW_all[i]
    
    # Direct: f(X_t) = X_t^2
    f_direct = X ** 2
    
    # Via Ito's Lemma: df = (2X*mu + sigma^2)dt + 2X*sigma*dW
    f_ito = np.zeros(n_steps + 1)
    f_ito[0] = x0 ** 2
    for i in range(n_steps):
        df = (2 * X[i] * mu + sigma**2) * dt + 2 * X[i] * sigma * dW_all[i]
        f_ito[i + 1] = f_ito[i] + df
    
    return X, f_direct, f_ito

In [ ]:
X, f_direct, f_ito = demonstrate_itos_lemma(1.0, 0.1, 0.3, 0.001, 1000)
t = np.linspace(0, 1.0, 1001)

plt.figure(figsize=(10, 4))
plt.plot(t, f_direct, label="Direct: X(t)²", linewidth=2)
plt.plot(t, f_ito, '--', label="Ito's Lemma reconstruction", linewidth=2)
plt.title("Ito's Lemma Demonstration: f(x) = x²")
plt.xlabel("Time")
plt.ylabel("f(X(t))")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

max_error = np.max(np.abs(f_direct - f_ito))
print(f"The maximum difference between direct computation and Ito's Lemma is {max_error:.6f}.")
print(f"The two curves nearly overlap, confirming that Ito's Lemma correctly captures the stochastic chain rule.")

### Mean-Reverting Process (Ornstein-Uhlenbeck)

A mean-reverting process tends to drift back toward a long-term average (mu). When the value
is above mu it gets pulled down; when below, it gets pulled up. The speed of mean reversion
is controlled by theta. Interest rates and volatility are often modeled this way because they
don't wander off to infinity like stock prices can.

In [ ]:
def simulate_ou_process(x0: float, theta: float, mu: float, sigma: float,
                         dt: float, n_steps: int) -> np.ndarray:
    """Simulate an Ornstein-Uhlenbeck (mean-reverting) process."""
    X = np.zeros(n_steps + 1)
    X[0] = x0
    for i in range(n_steps):
        dW = np.random.normal(0, np.sqrt(dt))
        X[i + 1] = X[i] + theta * (mu - X[i]) * dt + sigma * dW
    return X

In [ ]:
x0, theta, mu, sigma, dt, n_steps = 5.0, 2.0, 3.0, 0.5, 0.01, 1000
X_ou = simulate_ou_process(x0, theta, mu, sigma, dt, n_steps)
t = np.linspace(0, n_steps * dt, n_steps + 1)

plt.figure(figsize=(10, 4))
plt.plot(t, X_ou, label="OU Process")
plt.axhline(mu, color='red', linestyle='--', label=f"Long-term mean (μ={mu})")
plt.title("Ornstein-Uhlenbeck (Mean-Reverting) Process")
plt.xlabel("Time")
plt.ylabel("X(t)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Starting at X0={x0}, the process mean-reverts toward μ={mu} with speed θ={theta}.")
print(f"After T={n_steps*dt:.0f}, X(T)={X_ou[-1]:.2f}, which is close to the long-term mean.")

### Jump-Diffusion Process

Real markets sometimes experience sudden large moves (crashes, earnings surprises) that
a smooth diffusion process can't capture. A jump-diffusion adds random jumps (modeled by
a Poisson process) on top of the usual drift and Brownian motion. The jumps arrive randomly
and have a specified average size.

In [ ]:
def simulate_jump_diffusion(S0: float, mu: float, sigma: float, lam: float,
                             jump_mean: float, jump_std: float,
                             dt: float, n_steps: int) -> np.ndarray:
    """Simulate a jump-diffusion process (Merton model)."""
    S = np.zeros(n_steps + 1)
    S[0] = S0
    for i in range(n_steps):
        dW = np.random.normal(0, np.sqrt(dt))
        # Poisson jump
        n_jumps = np.random.poisson(lam * dt)
        J = np.sum(np.random.normal(jump_mean, jump_std, size=n_jumps)) if n_jumps > 0 else 0
        S[i + 1] = S[i] + mu * S[i] * dt + sigma * S[i] * dW + J * S[i]
    return S

In [ ]:
S0, mu, sigma = 100.0, 0.05, 0.2
lam, jump_mean, jump_std = 2.0, -0.02, 0.05  # ~2 jumps/year, slightly negative avg
dt, n_steps = 0.001, 1000

S_jd = simulate_jump_diffusion(S0, mu, sigma, lam, jump_mean, jump_std, dt, n_steps)
t = np.linspace(0, n_steps * dt, n_steps + 1)

plt.figure(figsize=(10, 4))
plt.plot(t, S_jd)
plt.title("Jump-Diffusion Process")
plt.xlabel("Time")
plt.ylabel("S(t)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Starting at S0=${S0}, with drift {mu*100}%, volatility {sigma*100}%, "
      f"and ~{lam} jumps/year, the price ended at ${S_jd[-1]:.2f} after T={n_steps*dt:.1f} years.")
print(f"Notice the occasional sudden moves — those are the Poisson-driven jumps.")

### Quadratic Variation

Quadratic variation measures the total accumulated "squared movement" of a stochastic process.
For a standard Brownian motion, the quadratic variation over [0, T] equals T — a remarkable
property that distinguishes random paths from smooth ones. Smooth functions have zero
quadratic variation; stochastic processes do not.

In [ ]:
def quadratic_variation(path: np.ndarray) -> float:
    """Compute the realized quadratic variation of a discrete path."""
    increments = np.diff(path)
    return np.sum(increments ** 2)

In [ ]:
# Simulate Brownian motion over T=1 and compute its quadratic variation
W_qv = simulate_brownian_motion(10000, 0.0001)  # fine grid
qv = quadratic_variation(W_qv)
T_total = 10000 * 0.0001
print(f"The quadratic variation of a Brownian motion over T={T_total} is {qv:.4f}.")
print(f"Theory predicts [W]_T = T = {T_total}, and our numerical estimate is very close.")

---
## 6. DEPENDENCE & STATISTICS
---

### PCA (Principal Component Analysis)

PCA finds the directions of maximum variance in multivariate data. In finance, if you have
returns of 50 stocks, PCA might reveal that most of the movement can be explained by 3-5
factors (like overall market, sector rotation, etc.). The eigenvalues tell you how much
variance each component explains.

In [ ]:
def pca_decomposition(data: np.ndarray) -> tuple:
    """Perform PCA via eigendecomposition of the covariance matrix."""
    cov_matrix = np.cov(data, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    # Sort descending
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    explained_variance_ratio = eigenvalues / np.sum(eigenvalues)
    return eigenvalues, eigenvectors, explained_variance_ratio

In [ ]:
# Simulate correlated 3-asset returns
cov = np.array([[0.04, 0.02, 0.01],
                [0.02, 0.09, 0.03],
                [0.01, 0.03, 0.16]])
returns_3d = np.random.multivariate_normal([0.01, 0.02, 0.015], cov, size=500)

eigenvalues, eigenvectors, evr = pca_decomposition(returns_3d)
for i, (ev, ratio) in enumerate(zip(eigenvalues, evr)):
    print(f"PC{i+1}: eigenvalue = {ev:.4f}, explains {ratio*100:.1f}% of total variance.")
print(f"\nThe first two components together explain {sum(evr[:2])*100:.1f}% of the variance, "
      f"meaning a 2-factor model captures most of the risk.")

### Kalman Filter (1D Scalar)

The Kalman filter estimates the true value of a hidden state from noisy measurements.
Imagine you're trying to track the true price of an illiquid asset, but each trade you observe
has noise. The Kalman filter optimally combines your prediction of where the price should be
with the noisy observation, weighting each by its uncertainty.

In [ ]:
def kalman_filter_1d(measurements: np.ndarray, x0: float, P0: float,
                      Q: float, R: float, A: float = 1.0, H: float = 1.0) -> np.ndarray:
    """Run a 1D scalar Kalman filter on a sequence of measurements."""
    n = len(measurements)
    x_est = np.zeros(n)
    x, P = x0, P0
    for i in range(n):
        # Predict
        x_pred = A * x
        P_pred = A * P * A + Q
        # Update
        K = P_pred * H / (H * P_pred * H + R)  # Kalman gain
        x = x_pred + K * (measurements[i] - H * x_pred)
        P = (1 - K * H) * P_pred
        x_est[i] = x
    return x_est

In [ ]:
# True signal + noise
true_signal = 50 + 0.1 * np.arange(100)  # slow uptrend
noise = np.random.normal(0, 3, size=100)
measurements = true_signal + noise

filtered = kalman_filter_1d(measurements, x0=50.0, P0=10.0, Q=0.1, R=9.0)

plt.figure(figsize=(10, 4))
plt.plot(measurements, 'o', markersize=2, alpha=0.5, label="Noisy measurements")
plt.plot(true_signal, '--', label="True signal", color='green')
plt.plot(filtered, label="Kalman estimate", color='red', linewidth=2)
plt.title("1D Kalman Filter")
plt.xlabel("Time step")
plt.ylabel("Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"The Kalman filter smooths out measurement noise while tracking the true trend.")
print(f"Final estimate: {filtered[-1]:.2f} vs true value: {true_signal[-1]:.2f} "
      f"(error: {abs(filtered[-1]-true_signal[-1]):.2f}).")

### Copula (Gaussian)

A copula separates the dependence structure between variables from their individual distributions.
A Gaussian copula uses the correlation matrix of normal distributions to generate correlated
uniform samples, which can then be transformed into any marginal distribution. This is widely
used in risk management to model joint defaults or correlated asset returns.

In [ ]:
def gaussian_copula_samples(rho: float, n_samples: int) -> tuple:
    """Generate correlated samples using a bivariate Gaussian copula."""
    mean = [0, 0]
    cov = [[1, rho], [rho, 1]]
    z = np.random.multivariate_normal(mean, cov, size=n_samples)
    u = norm.cdf(z)  # Transform to uniform [0,1] via CDF
    return u[:, 0], u[:, 1]

In [ ]:
u1, u2 = gaussian_copula_samples(rho=0.7, n_samples=2000)

plt.figure(figsize=(6, 6))
plt.scatter(u1, u2, s=2, alpha=0.4)
plt.title("Gaussian Copula Samples (ρ = 0.7)")
plt.xlabel("U1")
plt.ylabel("U2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Generated 2000 correlated uniform samples with Gaussian copula (ρ=0.7).")
print(f"The correlation between the uniform samples is {np.corrcoef(u1, u2)[0,1]:.3f}, "
      f"close to the input ρ=0.7.")

### CPPI (Constant Proportion Portfolio Insurance)

CPPI is a dynamic strategy that protects a minimum portfolio value (the floor) while still
participating in market gains. The idea: invest a multiple of the "cushion" (portfolio value
minus floor) in risky assets, and the rest in safe assets. As the portfolio grows, you invest
more aggressively; as it shrinks toward the floor, you become conservative.

In [ ]:
def simulate_cppi(initial_value: float, floor_pct: float, multiplier: float,
                   returns: np.ndarray) -> np.ndarray:
    """Simulate a CPPI strategy given a sequence of risky asset returns."""
    n = len(returns)
    portfolio = np.zeros(n + 1)
    portfolio[0] = initial_value
    floor_value = initial_value * floor_pct
    
    for i in range(n):
        cushion = max(portfolio[i] - floor_value, 0)
        risky_allocation = min(multiplier * cushion, portfolio[i])
        safe_allocation = portfolio[i] - risky_allocation
        portfolio[i + 1] = risky_allocation * (1 + returns[i]) + safe_allocation * 1.0002  # tiny safe return
    
    return portfolio

In [ ]:
risky_returns = np.random.normal(0.0005, 0.015, size=252)  # ~1 year of daily returns
portfolio = simulate_cppi(100_000, floor_pct=0.80, multiplier=3, returns=risky_returns)

plt.figure(figsize=(10, 4))
plt.plot(portfolio, label="CPPI Portfolio")
plt.axhline(80_000, color='red', linestyle='--', label="Floor ($80,000)")
plt.title("CPPI Strategy Simulation")
plt.xlabel("Trading Day")
plt.ylabel("Portfolio Value ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Starting with $100,000 and an $80,000 floor (80%), the CPPI strategy ended at ${portfolio[-1]:,.2f}.")
print(f"The strategy dynamically adjusted risk exposure to protect the floor while capturing upside.")

### APY (Annual Percentage Yield)

APY converts a nominal interest rate with compounding into the actual annual return you earn.
If a bank offers 5% compounded monthly, you actually earn slightly more than 5% because
each month's interest earns interest in subsequent months. APY lets you compare offers with
different compounding frequencies on equal footing.

In [ ]:
def apy(nominal_rate: float, compounding_periods: int) -> float:
    """Compute Annual Percentage Yield from nominal rate and compounding frequency."""
    return (1 + nominal_rate / compounding_periods) ** compounding_periods - 1

In [ ]:
r_nom, n_comp = 0.05, 12  # 5% nominal, monthly compounding
annual_yield = apy(r_nom, n_comp)
print(f"A nominal rate of {r_nom*100}% compounded {n_comp} times per year gives an APY of {annual_yield*100:.4f}%.")
print(f"The extra {(annual_yield - r_nom)*100:.4f}% comes from earning interest on interest.")

### Capital Efficiency

Capital efficiency measures how much return you generate per unit of capital deployed.
A capital efficiency of 0.25 means every dollar of capital produced $0.25 in returns.
Traders and DeFi protocols use this metric to compare how productively capital is being used.

In [ ]:
def capital_efficiency(returns: float, capital_used: float) -> float:
    """Compute capital efficiency as return divided by capital deployed."""
    return returns / capital_used

In [ ]:
ret, cap = 50_000, 200_000
ce = capital_efficiency(ret, cap)
print(f"Generating ${ret:,} in returns on ${cap:,} of capital gives a capital efficiency of {ce*100:.1f}%.")
print(f"Every dollar of capital produced ${ce:.2f} in returns.")

---
## 7. DeFi / AMM
---

### Impermanent Loss

Impermanent loss happens when you provide liquidity to an automated market maker (AMM) and
the price of one asset changes relative to when you deposited. The pool automatically rebalances,
so you end up with more of the cheaper asset and less of the expensive one. The "loss" is
compared to simply holding the tokens — it is called impermanent because it reverses if the
price returns to its original level.

In [ ]:
def impermanent_loss(price_ratio: float) -> float:
    """Compute impermanent loss given the price ratio (current / entry)."""
    return 2 * np.sqrt(price_ratio) / (1 + price_ratio) - 1

In [ ]:
price_ratios = np.linspace(0.1, 5.0, 200)
il_values = [impermanent_loss(p) * 100 for p in price_ratios]

plt.figure(figsize=(10, 4))
plt.plot(price_ratios, il_values)
plt.axhline(0, color='gray', linestyle='--')
plt.title("Impermanent Loss vs. Price Ratio")
plt.xlabel("Price Ratio (current / entry)")
plt.ylabel("Impermanent Loss (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Specific example
p = 2.0
il = impermanent_loss(p)
print(f"If the price doubles (ratio = {p}), the impermanent loss is {il*100:.2f}%.")
print(f"You would have been {abs(il)*100:.2f}% better off just holding the tokens instead of providing liquidity.")

### Health Factor

In DeFi lending protocols, the health factor tells you how safe your position is from
liquidation. It is the ratio of your collateral (adjusted by the liquidation threshold)
to your debt. A health factor above 1 means you are safe; below 1 means your position
can be liquidated. Borrowers should monitor this closely.

In [ ]:
def health_factor(collateral: float, liquidation_threshold: float, debt: float) -> float:
    """Compute the health factor for a DeFi lending position."""
    return (collateral * liquidation_threshold) / debt

In [ ]:
coll, liq_thresh, debt = 10_000, 0.80, 5_000
hf = health_factor(coll, liq_thresh, debt)
print(f"With ${coll:,} collateral (80% liquidation threshold) and ${debt:,} debt, "
      f"the health factor is {hf:.2f}.")
print(f"Since HF={hf:.2f} > 1.0, the position is safe from liquidation.")
print(f"The collateral would need to drop to ${debt/liq_thresh:,.0f} before liquidation triggers.")

### Borrow Rate (Kinked Utilization Model)

DeFi lending protocols set borrow rates based on how much of the lending pool is being used
(utilization). A kinked model keeps rates low when utilization is moderate, but rates spike
sharply above a threshold (the kink) to incentivize repayment and discourage over-borrowing.
This protects depositors by ensuring the pool doesn't get fully drained.

In [ ]:
def borrow_rate(utilization: float, base_rate: float = 0.02,
                 slope1: float = 0.04, slope2: float = 0.75, kink: float = 0.80) -> float:
    """Compute borrow rate using a kinked utilization model."""
    if utilization <= kink:
        return base_rate + slope1 * (utilization / kink)
    else:
        rate_at_kink = base_rate + slope1
        excess = (utilization - kink) / (1 - kink)
        return rate_at_kink + slope2 * excess

In [ ]:
utilizations = np.linspace(0, 1, 200)
rates = [borrow_rate(u) * 100 for u in utilizations]

plt.figure(figsize=(10, 4))
plt.plot(utilizations * 100, rates)
plt.axvline(80, color='red', linestyle='--', alpha=0.5, label="Kink (80%)")
plt.title("Borrow Rate vs. Utilization (Kinked Model)")
plt.xlabel("Utilization (%)")
plt.ylabel("Borrow Rate (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

u_example = 0.90
r_example = borrow_rate(u_example)
print(f"At {u_example*100:.0f}% utilization, the borrow rate is {r_example*100:.2f}%.")
print(f"Rates spike above the 80% kink to discourage further borrowing and protect depositors.")

### TVL (Total Value Locked)

TVL is the total dollar value of all assets deposited in a DeFi protocol. It is computed
by summing the quantity of each asset multiplied by its current price. TVL is the most
common metric for measuring the size and adoption of a DeFi protocol.

In [ ]:
def total_value_locked(quantities: np.ndarray, prices: np.ndarray) -> float:
    """Compute total value locked as the sum of quantity * price for each asset."""
    return np.sum(quantities * prices)

In [ ]:
quantities = np.array([500, 10_000_000, 1_000])   # ETH, USDC, BTC
prices = np.array([3_200, 1.0, 65_000])            # current prices
tvl = total_value_locked(quantities, prices)
print(f"The protocol holds 500 ETH, 10M USDC, and 1,000 BTC.")
print(f"Total Value Locked = ${tvl:,.0f} ({tvl/1e6:.1f}M).")
print(f"This gives a snapshot of how much capital is trusted to this protocol.")

---
## 8. TRADING / EXECUTION
---

### Slippage

Slippage is the difference between the price you expected to get and the price you actually
got when your order was executed. It usually happens because the market moved between the time
you placed the order and when it was filled, or because the order book didn't have enough
liquidity at the expected price.

In [ ]:
def slippage(execution_price: float, expected_price: float) -> float:
    """Compute slippage as the difference between execution and expected price."""
    return execution_price - expected_price

In [ ]:
p_exec, p_exp = 101.50, 101.00
slip = slippage(p_exec, p_exp)
slip_bps = (slip / p_exp) * 10000
print(f"Expected to buy at ${p_exp}, but actually paid ${p_exec}.")
print(f"The slippage is ${slip:.2f} ({slip_bps:.1f} basis points), meaning the trade cost "
      f"more than planned due to market movement or low liquidity.")

### Price Impact

Price impact is how much your own order moves the market price. Large orders push the price
because they consume available liquidity at the best prices. The market impact coefficient (lambda)
captures how sensitive the market is — illiquid markets have higher lambda, meaning even
small orders cause significant price moves.

In [ ]:
def price_impact(lam: float, order_size: float) -> float:
    """Compute linear price impact: delta_P = lambda * Q."""
    return lam * order_size

In [ ]:
lam, Q = 0.001, 5000  # lambda = $0.001 per share, order = 5000 shares
dp = price_impact(lam, Q)
print(f"Buying {Q:,} shares with a market impact coefficient of λ={lam} causes "
      f"a price impact of ${dp:.2f}.")
print(f"This means the stock price is expected to move up by ${dp:.2f} as a result of the order.")